# Part 1: Environment Setup & Data Preparation

This notebook covers:
- Environment setup and package installation for Kaggle P100 GPU
- Global configuration and paths
- COCO dataset classes for TD and TSR
- Augmentation pipeline from augmentation_config.json

**Models:**
- TD: `microsoft/table-transformer-detection`
- TSR: `microsoft/table-transformer-structure-recognition`

**Reference:** Tablecert: YOLO and TATR Enhanced Models to Boost Table Detection and Recognition in Legacy Documents

## 1.1 Install Dependencies

In [ ]:
%%capture
!pip install -q transformers>=4.35.0
!pip install -q peft>=0.7.0
!pip install -q albumentations>=1.3.0
!pip install -q pycocotools>=2.0.6
!pip install -q torchmetrics>=1.0.0
!pip install -q timm>=0.9.0
!pip install -q accelerate>=0.25.0
!pip install -q datasets>=2.14.0
!pip install -q matplotlib seaborn pandas
print("All dependencies installed successfully!")

## 1.2 Import Libraries

In [ ]:
import os
import json
import random
import numpy as np
from pathlib import Path
from typing import Dict, List, Tuple, Optional, Any
from dataclasses import dataclass, field
from collections import defaultdict
import warnings
warnings.filterwarnings('ignore')

# PyTorch
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torch.optim import AdamW
from torch.optim.lr_scheduler import CosineAnnealingLR
from torch.cuda.amp import autocast, GradScaler

# Transformers & PEFT
from transformers import (
    AutoModelForObjectDetection,
    AutoImageProcessor,
    DetrImageProcessor,
    TableTransformerForObjectDetection,
)
from peft import LoraConfig, get_peft_model, TaskType

# Image processing
from PIL import Image
import albumentations as A
from albumentations.pytorch import ToTensorV2

# Evaluation
from pycocotools.coco import COCO
from pycocotools.cocoeval import COCOeval

# Visualization
import matplotlib.pyplot as plt
import matplotlib.patches as patches
import seaborn as sns
import pandas as pd
from tqdm.auto import tqdm

print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")

## 1.3 Global Configuration

In [ ]:
@dataclass
class Config:
    """Global configuration for TATR TD and TSR training."""
    
    # Paths - Kaggle dataset paths
    IMAGES_DIR: str = "/kaggle/input/datasets/mohammedahmedxx12/machathon-dataset/Phase1_Train_Dataset/images"
    TD_ANNOTATIONS_DIR: str = "/kaggle/input/datasets/philopaterg/stp26-preprocessed-dataset/preprocessed_dataset/table_detection"
    TSR_ANNOTATIONS_DIR: str = "/kaggle/input/datasets/philopaterg/stp26-preprocessed-dataset/preprocessed_dataset/table_structure"
    AUGMENTATION_CONFIG: str = "/kaggle/input/datasets/philopaterg/stp26-preprocessed-dataset/preprocessed_dataset/table_detection/augmentation_config.json"
    OUTPUT_DIR: str = "/kaggle/working/outputs"
    
    # Model identifiers
    TD_MODEL_NAME: str = "microsoft/table-transformer-detection"
    TSR_MODEL_NAME: str = "microsoft/table-transformer-structure-recognition"
    
    # Training hyperparameters (from Tablecert paper)
    LEARNING_RATE_LORA: float = 1e-3
    LEARNING_RATE_NO_LORA: float = 5e-5
    BATCH_SIZE: int = 4  # Per device, suitable for P100 16GB
    GRADIENT_ACCUMULATION_STEPS: int = 4
    MAX_EPOCHS: int = 50
    PATIENCE: int = 10
    WEIGHT_DECAY: float = 1e-5
    MAX_GRAD_NORM: float = 0.01
    
    # Image size
    IMAGE_SIZE: int = 800
    
    # LoRA configuration (from Tablecert paper)
    LORA_R: int = 16
    LORA_ALPHA: int = 32
    LORA_DROPOUT: float = 0.05
    
    # TSR categories
    TSR_CATEGORIES: List[str] = field(default_factory=lambda: [
        "table column",
        "table row", 
        "table column header",
        "table projected row header",
        "table spanning cell"
    ])
    
    # TD categories  
    TD_CATEGORIES: List[str] = field(default_factory=lambda: ["table"])
    
    # Random seed
    SEED: int = 42
    
    # Device
    DEVICE: str = "cuda" if torch.cuda.is_available() else "cpu"
    
    # Mixed precision
    USE_AMP: bool = True


config = Config()

# Create output directory
os.makedirs(config.OUTPUT_DIR, exist_ok=True)

print("Configuration loaded:")
print(f"  Device: {config.DEVICE}")
print(f"  Image size: {config.IMAGE_SIZE}")
print(f"  Batch size: {config.BATCH_SIZE}")
print(f"  LoRA rank: {config.LORA_R}")

In [ ]:
def set_seed(seed: int):
    """Set random seeds for reproducibility."""
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
        torch.backends.cudnn.deterministic = True
        torch.backends.cudnn.benchmark = False

set_seed(config.SEED)
print(f"Random seed set to {config.SEED}")

## 1.4 Augmentation Pipeline

Build augmentation pipeline from `augmentation_config.json` using Albumentations with bbox-safe transforms.

In [ ]:
def load_augmentation_config(config_path: str) -> Dict:
    """Load augmentation configuration from JSON file."""
    with open(config_path, 'r') as f:
        return json.load(f)


def build_augmentation_pipeline(aug_config: Dict, is_train: bool = True) -> A.Compose:
    """
    Build Albumentations augmentation pipeline from config.
    
    Args:
        aug_config: Augmentation configuration dictionary
        is_train: Whether to apply training augmentations
    
    Returns:
        Albumentations Compose pipeline
    """
    transforms = []
    
    if is_train:
        # Geometric transforms
        geo = aug_config.get('geometric', {})
        
        # Horizontal flip
        if geo.get('horizontal_flip', {}).get('enabled', False):
            transforms.append(A.HorizontalFlip(
                p=geo['horizontal_flip'].get('probability', 0.3)
            ))
        
        # Rotation
        if geo.get('rotation', {}).get('enabled', False):
            transforms.append(A.Rotate(
                limit=geo['rotation'].get('limit_degrees', 5),
                border_mode=0,  # cv2.BORDER_CONSTANT
                value=(255, 255, 255),
                p=geo['rotation'].get('probability', 0.2)
            ))
        
        # Perspective
        if geo.get('perspective', {}).get('enabled', False):
            scale = geo['perspective'].get('scale', [0.02, 0.05])
            transforms.append(A.Perspective(
                scale=(scale[0], scale[1]),
                p=geo['perspective'].get('probability', 0.15)
            ))
        
        # Affine
        if geo.get('affine', {}).get('enabled', False):
            aff = geo['affine']
            transforms.append(A.Affine(
                scale=(aff.get('scale', [0.95, 1.05])[0], aff.get('scale', [0.95, 1.05])[1]),
                translate_percent={'x': (-aff.get('translate_percent', 0.02), aff.get('translate_percent', 0.02)),
                                   'y': (-aff.get('translate_percent', 0.02), aff.get('translate_percent', 0.02))},
                shear={'x': tuple(aff.get('shear', [-2, 2])), 'y': tuple(aff.get('shear', [-2, 2]))},
                p=aff.get('probability', 0.15)
            ))
        
        # Photometric transforms
        photo = aug_config.get('photometric', {})
        
        # Brightness/Contrast
        if photo.get('brightness_contrast', {}).get('enabled', False):
            bc = photo['brightness_contrast']
            transforms.append(A.RandomBrightnessContrast(
                brightness_limit=bc.get('brightness_limit', 0.15),
                contrast_limit=bc.get('contrast_limit', 0.15),
                p=bc.get('probability', 0.4)
            ))
        
        # HSV shift
        if photo.get('hsv_shift', {}).get('enabled', False):
            hsv = photo['hsv_shift']
            transforms.append(A.HueSaturationValue(
                hue_shift_limit=hsv.get('hue_shift_limit', 10),
                sat_shift_limit=hsv.get('sat_shift_limit', 15),
                val_shift_limit=hsv.get('val_shift_limit', 15),
                p=hsv.get('probability', 0.3)
            ))
        
        # Grayscale
        if photo.get('grayscale', {}).get('enabled', False):
            transforms.append(A.ToGray(
                p=photo['grayscale'].get('probability', 0.1)
            ))
        
        # Blur
        if photo.get('blur', {}).get('enabled', False):
            transforms.append(A.GaussianBlur(
                blur_limit=photo['blur'].get('blur_limit', 3),
                p=photo['blur'].get('probability', 0.2)
            ))
        
        # Gaussian noise
        noise = photo.get('noise', {})
        if noise.get('gauss_noise', {}).get('enabled', False):
            gn = noise['gauss_noise']
            transforms.append(A.GaussNoise(
                var_limit=tuple(gn.get('var_limit', [5, 20])),
                p=gn.get('probability', 0.15)
            ))
        
        # ISO noise
        if noise.get('iso_noise', {}).get('enabled', False):
            iso = noise['iso_noise']
            transforms.append(A.ISONoise(
                intensity=tuple(iso.get('intensity', [0.1, 0.3])),
                p=iso.get('probability', 0.1)
            ))
        
        # JPEG compression
        if photo.get('jpeg_compression', {}).get('enabled', False):
            jpeg = photo['jpeg_compression']
            transforms.append(A.ImageCompression(
                quality_lower=jpeg.get('quality_lower', 50),
                quality_upper=jpeg.get('quality_upper', 95),
                p=jpeg.get('probability', 0.2)
            ))
        
        # Document-specific transforms
        doc = aug_config.get('document_specific', {})
        
        # Grid distortion
        if doc.get('grid_distortion', {}).get('enabled', False):
            gd = doc['grid_distortion']
            transforms.append(A.GridDistortion(
                distort_limit=gd.get('distort_limit', 0.1),
                p=gd.get('probability', 0.1)
            ))
    
    # Get bbox params from config
    compose_cfg = aug_config.get('compose', {})
    bbox_params = A.BboxParams(
        format=compose_cfg.get('bbox_format', 'coco'),
        min_area=compose_cfg.get('min_area', 100),
        min_visibility=compose_cfg.get('min_visibility', 0.3),
        label_fields=compose_cfg.get('label_fields', ['category_id'])
    )
    
    return A.Compose(transforms, bbox_params=bbox_params)


# Load augmentation config
try:
    aug_config = load_augmentation_config(config.AUGMENTATION_CONFIG)
    print("Augmentation config loaded successfully")
    print(f"  Geometric transforms: {list(aug_config.get('geometric', {}).keys())}")
    print(f"  Photometric transforms: {list(aug_config.get('photometric', {}).keys())}")
except FileNotFoundError:
    # Fallback default config for local testing
    aug_config = {
        "geometric": {
            "horizontal_flip": {"enabled": True, "probability": 0.3},
            "rotation": {"enabled": True, "probability": 0.2, "limit_degrees": 5},
            "perspective": {"enabled": True, "probability": 0.15, "scale": [0.02, 0.05]},
            "affine": {"enabled": True, "probability": 0.15, "scale": [0.95, 1.05], "translate_percent": 0.02, "shear": [-2, 2]}
        },
        "photometric": {
            "brightness_contrast": {"enabled": True, "probability": 0.4, "brightness_limit": 0.15, "contrast_limit": 0.15},
            "hsv_shift": {"enabled": True, "probability": 0.3, "hue_shift_limit": 10, "sat_shift_limit": 15, "val_shift_limit": 15},
            "grayscale": {"enabled": True, "probability": 0.1},
            "blur": {"enabled": True, "probability": 0.2, "blur_limit": 3},
            "noise": {
                "gauss_noise": {"enabled": True, "probability": 0.15, "var_limit": [5, 20]},
                "iso_noise": {"enabled": True, "probability": 0.1, "intensity": [0.1, 0.3]}
            },
            "jpeg_compression": {"enabled": True, "probability": 0.2, "quality_lower": 50, "quality_upper": 95}
        },
        "document_specific": {
            "grid_distortion": {"enabled": True, "probability": 0.1, "distort_limit": 0.1}
        },
        "compose": {
            "bbox_format": "coco",
            "min_area": 100,
            "min_visibility": 0.3,
            "label_fields": ["category_id"]
        }
    }
    print("Using default augmentation config")

# Build pipelines
train_transform = build_augmentation_pipeline(aug_config, is_train=True)
val_transform = build_augmentation_pipeline(aug_config, is_train=False)
print(f"\nTraining augmentation pipeline built with {len(train_transform.transforms)} transforms")

## 1.5 COCO Dataset Classes

In [ ]:
class COCOTableDataset(Dataset):
    """
    COCO-format dataset for Table Detection and Table Structure Recognition.
    
    Supports:
    - COCO JSON annotation format
    - Albumentations augmentation pipeline
    - HuggingFace DetrImageProcessor preprocessing
    """
    
    def __init__(
        self,
        annotations_file: str,
        images_dir: str,
        processor: DetrImageProcessor,
        augmentation: Optional[A.Compose] = None,
        task: str = "detection",  # 'detection' for TD, 'structure' for TSR
    ):
        """
        Initialize the dataset.
        
        Args:
            annotations_file: Path to COCO JSON annotations
            images_dir: Path to images directory
            processor: HuggingFace image processor
            augmentation: Albumentations compose pipeline
            task: 'detection' for TD or 'structure' for TSR
        """
        self.images_dir = images_dir
        self.processor = processor
        self.augmentation = augmentation
        self.task = task
        
        # Load COCO annotations
        with open(annotations_file, 'r') as f:
            self.coco_data = json.load(f)
        
        # Build image id to annotations mapping
        self.images = {img['id']: img for img in self.coco_data['images']}
        self.categories = {cat['id']: cat['name'] for cat in self.coco_data['categories']}
        self.num_classes = len(self.categories)
        
        # Group annotations by image
        self.img_to_anns = defaultdict(list)
        for ann in self.coco_data['annotations']:
            self.img_to_anns[ann['image_id']].append(ann)
        
        # Only keep images with annotations
        self.image_ids = [img_id for img_id in self.images.keys() 
                         if len(self.img_to_anns[img_id]) > 0]
        
        print(f"Loaded {len(self.image_ids)} images with {len(self.coco_data['annotations'])} annotations")
        print(f"Categories: {self.categories}")
    
    def __len__(self) -> int:
        return len(self.image_ids)
    
    def __getitem__(self, idx: int) -> Dict[str, Any]:
        image_id = self.image_ids[idx]
        image_info = self.images[image_id]
        annotations = self.img_to_anns[image_id]
        
        # Load image
        image_path = os.path.join(self.images_dir, image_info['file_name'])
        image = Image.open(image_path).convert('RGB')
        image_np = np.array(image)
        
        # Extract bboxes and labels
        bboxes = []
        labels = []
        areas = []
        
        for ann in annotations:
            # COCO format: [x, y, width, height]
            bbox = ann['bbox']
            if bbox[2] > 0 and bbox[3] > 0:  # Valid bbox
                bboxes.append(bbox)
                # Adjust category_id (COCO is 1-indexed, model expects 0-indexed)
                labels.append(ann['category_id'] - 1)
                areas.append(ann.get('area', bbox[2] * bbox[3]))
        
        # Apply augmentation
        if self.augmentation is not None and len(bboxes) > 0:
            try:
                augmented = self.augmentation(
                    image=image_np,
                    bboxes=bboxes,
                    category_id=labels
                )
                image_np = augmented['image']
                bboxes = augmented['bboxes']
                labels = augmented['category_id']
            except Exception as e:
                # If augmentation fails, use original
                pass
        
        # Convert image back to PIL for processor
        image = Image.fromarray(image_np)
        
        # Convert COCO format [x, y, w, h] to DETR format [x_center, y_center, w, h] normalized
        h, w = image_np.shape[:2]
        target_boxes = []
        target_labels = []
        
        for bbox, label in zip(bboxes, labels):
            x, y, bw, bh = bbox
            # Convert to center format and normalize
            x_center = (x + bw / 2) / w
            y_center = (y + bh / 2) / h
            norm_w = bw / w
            norm_h = bh / h
            target_boxes.append([x_center, y_center, norm_w, norm_h])
            target_labels.append(label)
        
        # Prepare target
        target = {
            'boxes': torch.tensor(target_boxes, dtype=torch.float32) if target_boxes else torch.zeros((0, 4)),
            'class_labels': torch.tensor(target_labels, dtype=torch.int64) if target_labels else torch.zeros((0,), dtype=torch.int64),
            'image_id': torch.tensor([image_id]),
            'orig_size': torch.tensor([h, w]),
        }
        
        # Process with HuggingFace processor
        encoding = self.processor(
            images=image,
            annotations=[{
                'boxes': target_boxes,
                'class_labels': target_labels,
            }] if target_boxes else None,
            return_tensors='pt'
        )
        
        # Remove batch dimension
        pixel_values = encoding['pixel_values'].squeeze(0)
        
        return {
            'pixel_values': pixel_values,
            'labels': target,
            'image_id': image_id,
        }


def collate_fn(batch: List[Dict]) -> Dict[str, Any]:
    """Custom collate function for DataLoader."""
    pixel_values = torch.stack([item['pixel_values'] for item in batch])
    labels = [item['labels'] for item in batch]
    image_ids = [item['image_id'] for item in batch]
    
    return {
        'pixel_values': pixel_values,
        'labels': labels,
        'image_ids': image_ids,
    }

## 1.6 Load Datasets

In [ ]:
def load_datasets(config: Config, task: str = "detection"):
    """
    Load train, validation, and test datasets.
    
    Args:
        config: Configuration object
        task: 'detection' for TD or 'structure' for TSR
    
    Returns:
        Tuple of (train_dataset, val_dataset, test_dataset, processor)
    """
    # Select model and paths based on task
    if task == "detection":
        model_name = config.TD_MODEL_NAME
        annotations_dir = config.TD_ANNOTATIONS_DIR
        images_dir = config.IMAGES_DIR
    else:
        model_name = config.TSR_MODEL_NAME
        annotations_dir = config.TSR_ANNOTATIONS_DIR
        # TSR uses cropped table images - they should be in a specific directory
        # For now, we'll point to the same images dir (will be converted in preprocessing)
        images_dir = config.IMAGES_DIR
    
    # Load processor
    processor = DetrImageProcessor.from_pretrained(
        model_name,
        size={"shortest_edge": config.IMAGE_SIZE, "longest_edge": config.IMAGE_SIZE},
        do_resize=True,
        do_normalize=True,
    )
    
    # Build augmentation pipeline
    train_aug = build_augmentation_pipeline(aug_config, is_train=True)
    val_aug = build_augmentation_pipeline(aug_config, is_train=False)
    
    # Load datasets
    train_file = os.path.join(annotations_dir, "train.json")
    val_file = os.path.join(annotations_dir, "val.json")
    test_file = os.path.join(annotations_dir, "test.json")
    
    print(f"\nLoading {task} datasets...")
    
    datasets = {}
    
    for split, (file_path, aug) in [
        ('train', (train_file, train_aug)),
        ('val', (val_file, val_aug)),
        ('test', (test_file, val_aug)),
    ]:
        if os.path.exists(file_path):
            print(f"\n{split.upper()} split:")
            datasets[split] = COCOTableDataset(
                annotations_file=file_path,
                images_dir=images_dir,
                processor=processor,
                augmentation=aug if split == 'train' else None,
                task=task,
            )
        else:
            print(f"Warning: {file_path} not found")
            datasets[split] = None
    
    return datasets.get('train'), datasets.get('val'), datasets.get('test'), processor


# This will be called in subsequent notebooks
print("Dataset loading functions defined.")
print("Use load_datasets(config, task='detection') for TD")
print("Use load_datasets(config, task='structure') for TSR")

## 1.7 Utility Functions

In [ ]:
def visualize_sample(dataset: COCOTableDataset, idx: int = 0, figsize: Tuple[int, int] = (12, 8)):
    """
    Visualize a sample from the dataset with bounding boxes.
    
    Args:
        dataset: COCOTableDataset instance
        idx: Sample index
        figsize: Figure size
    """
    sample = dataset[idx]
    pixel_values = sample['pixel_values']
    labels = sample['labels']
    
    # Denormalize image
    mean = torch.tensor([0.485, 0.456, 0.406]).view(3, 1, 1)
    std = torch.tensor([0.229, 0.224, 0.225]).view(3, 1, 1)
    image = pixel_values * std + mean
    image = image.permute(1, 2, 0).numpy()
    image = np.clip(image, 0, 1)
    
    fig, ax = plt.subplots(1, 1, figsize=figsize)
    ax.imshow(image)
    
    h, w = image.shape[:2]
    boxes = labels['boxes']
    class_labels = labels['class_labels']
    
    colors = plt.cm.Set3(np.linspace(0, 1, dataset.num_classes))
    
    for box, cls in zip(boxes, class_labels):
        # Convert from center format to corner format
        x_center, y_center, bw, bh = box.tolist()
        x = (x_center - bw / 2) * w
        y = (y_center - bh / 2) * h
        box_w = bw * w
        box_h = bh * h
        
        color = colors[cls.item() % len(colors)]
        rect = patches.Rectangle(
            (x, y), box_w, box_h,
            linewidth=2, edgecolor=color, facecolor='none'
        )
        ax.add_patch(rect)
        
        # Add label
        category_name = dataset.categories.get(cls.item() + 1, f"Class {cls.item()}")
        ax.text(x, y - 5, category_name, fontsize=8, color=color,
                bbox=dict(boxstyle='round', facecolor='white', alpha=0.7))
    
    ax.set_title(f"Sample {idx} - Image ID: {sample['image_id']}")
    ax.axis('off')
    plt.tight_layout()
    plt.show()


print("Utility functions defined.")

## 1.8 Save Configuration for Other Notebooks

In [ ]:
# Save configuration and augmentation config for use in other notebooks
import pickle

config_to_save = {
    'config': config,
    'aug_config': aug_config,
}

config_path = os.path.join(config.OUTPUT_DIR, 'notebook_config.pkl')
with open(config_path, 'wb') as f:
    pickle.dump(config_to_save, f)

print(f"Configuration saved to {config_path}")
print("\n" + "="*60)
print("Part 1 Complete - Environment and Data Preparation Done")
print("="*60)
print("\nNext: Run Part 2 for TD Baseline Inference")